In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 3, Finished, Available, Finished, False)

SILVER

In [2]:
#read table trips
df_trips = spark.read.table("trips_bronze")
df_trips.printSchema()
df_trips.show(5, truncate=False)

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 4, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [4]:
#snake_case
df_trips.columns
df_trips = df_trips.withColumnRenamed("VendorID", "vendor_id") \
    .withColumnRenamed("RatecodeID", "ratecode_id") \
    .withColumnRenamed("PULocationID", "pu_location_id") \
    .withColumnRenamed("DOLocationID", "do_location_id") \
    .withColumnRenamed("Airport_fee", "airport_fee")

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 6, Finished, Available, Finished, False)

In [6]:
#search duplicates
total = df_trips.count()
distintos = df_trips.distinct().count()
print(f"Total: {total}, Distintos: {distintos}, Duplicados: {total - distintos}") 

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 8, Finished, Available, Finished, False)

Total: 7124755, Distintos: 7124755, Duplicados: 0


In [8]:
#search null 
print("pickup_datetime nulls:", df_trips.filter(F.col("tpep_pickup_datetime").isNull()).count())
print("dropoff_datetime nulls:", df_trips.filter(F.col("tpep_dropoff_datetime").isNull()).count())
print("pu_location_id nulls:", df_trips.filter(F.col("pu_location_id").isNull()).count())
print("do_location_id nulls:", df_trips.filter(F.col("do_location_id").isNull()).count())
print("total_amount nulls:", df_trips.filter(F.col("total_amount").isNull()).count())

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 10, Finished, Available, Finished, False)

pickup_datetime nulls: 0
dropoff_datetime nulls: 0
pu_location_id nulls: 0
do_location_id nulls: 0
total_amount nulls: 0


In [9]:
#data type
df_trips.printSchema()

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 11, Finished, Available, Finished, False)

root
 |-- vendor_id: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecode_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [11]:
#check consitence
df_trips.select("store_and_fwd_flag").distinct().show()

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 13, Finished, Available, Finished, False)

+------------------+
|store_and_fwd_flag|
+------------------+
|                 Y|
|                 N|
|              NULL|
+------------------+



In [12]:
#replace NULL with N
df_trips = df_trips.na.fill({"store_and_fwd_flag": "N"})

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 14, Finished, Available, Finished, False)

In [13]:
#new columns
df_trips = df_trips.withColumn("trip_duration_min", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 15, Finished, Available, Finished, False)

In [15]:
#check registers
print("Duración negativa o cero:", df_trips.filter(F.col("trip_duration_min") <= 0).count())
print("Total negativo:", df_trips.filter(F.col("total_amount") <= 0).count())
print("Distancia cero:", df_trips.filter(F.col("trip_distance") <= 0).count())
print("Pasajeros cero:", df_trips.filter(F.col("passenger_count") == 0).count())

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 17, Finished, Available, Finished, False)

Duración negativa o cero: 85668
Total negativo: 68191
Distancia cero: 248988
Pasajeros cero: 27671


In [21]:
#filters
df_trips = df_trips.filter(~((F.col("trip_distance") == 0) & (F.col("trip_duration_min") <= 0)))    
df_trips = df_trips.filter(
    (F.col("trip_duration_min") > 0) & 
    (F.col("passenger_count") > 0)
)
df_trips = df_trips.filter(F.col("total_amount") > 0)

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 23, Finished, Available, Finished, False)

In [22]:
#check registers
print(f"Filas después de limpiar: {df_trips.count()}")

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 24, Finished, Available, Finished, False)

Filas después de limpiar: 4832412


In [23]:
#quality 
print(f"Total filas: {df_trips.count()}")
df_trips.describe().show()
df_trips.printSchema()

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 25, Finished, Available, Finished, False)

Total filas: 4832412
+-------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+-----------------+--------------------+-------------------+------------------+--------------------+
|summary|          vendor_id|   passenger_count|     trip_distance|       ratecode_id|store_and_fwd_flag|    pu_location_id|    do_location_id|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|     total_amount|congestion_surcharge|        airport_fee|cbd_congestion_fee|   trip_duration_min|
+-------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------

In [24]:
#correct
df_trips = df_trips.filter(F.col("trip_distance") <= 620) \
    .filter(F.col("trip_duration_min") <= 1440) \
    .filter(F.col("total_amount") <= 1000) \
    .withColumn("ratecode_id", F.when(F.col("ratecode_id") == 99, 1).otherwise(F.col("ratecode_id")))

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 26, Finished, Available, Finished, False)

In [25]:
print(f"Filas después de limpiar: {df_trips.count()}")

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 27, Finished, Available, Finished, False)

Filas después de limpiar: 4832344


In [26]:
#save as silver
df_trips.write.format("delta").mode("overwrite").saveAsTable("trips_silver")


StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 28, Finished, Available, Finished, False)

In [27]:
#zones
df_zones = spark.read.table("zones_bronze")
df_zones.show(5)
df_zones.count()
df_zones.printSchema()

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 29, Finished, Available, Finished, False)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [28]:
#snake_case
df_zones = df_zones.withColumnRenamed("LocationID", "location_id") \
    .withColumnRenamed("Borough", "borough") \
    .withColumnRenamed("Zone", "zone")

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 30, Finished, Available, Finished, False)

In [30]:
#data type 
df_zones.withColumn("location_id", F.col("location_id").cast(IntegerType()))

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 32, Finished, Available, Finished, False)

DataFrame[location_id: int, borough: string, zone: string, service_zone: string]

In [31]:
#save zones as silver
df_zones.write.format("delta").mode("overwrite").saveAsTable("zones_silver")

StatementMeta(, 5224853a-cffb-48b4-b73f-aee9c9dd67d5, 33, Finished, Available, Finished, False)